# Pemilihan Algoritma Regresi pada Dataset E-Commerce
Target: **`net_sales`** | Dataset: `ecommerce_sales_customer_analytics_150k.csv`

Notebook ini berisi (1) landasan teori pemilihan algoritma regresi, (2) diagnosis 3 faktor penentu (jenis data, ukuran dataset, linearitas), (3) perbandingan banyak algoritma dengan MAE, MSE, RMSE, R², dan waktu prediksi, serta (4) panduan peruntukan tiap algoritma.

> Jika file CSV tidak ditemukan, notebook otomatis memakai **data sintetis** supaya seluruh sel tetap bisa dijalankan. Letakkan CSV Anda di folder yang sama untuk memakai data asli.

## 1. Teori Singkat

### 1.1 Tiga faktor utama pemilihan algoritma regresi
| Faktor | Pertanyaan | Dampak pada pilihan |
|---|---|---|
| **Jenis data** | Numerik, kategorikal, campuran? Ada missing value / outlier? Banyak fitur tidak relevan? | Data campuran + kategorikal: model berbasis pohon menangani lebih natural. Model linear/KNN/SVR/NN butuh scaling dan encoding. |
| **Ukuran dataset** | Ratusan, ribuan, atau ratusan ribu baris? Berapa jumlah fitur? | Data kecil: model sederhana/teregularisasi (Ridge, Lasso) agar tidak overfit. Data besar: boosting, Random Forest, SGD, atau neural net. SVR dan KNN makin berat saat data membesar. |
| **Linearitas** | Hubungan fitur-target mendekati garis lurus atau melengkung/berinteraksi? | Linear: Linear/Ridge/Lasso sudah cukup dan mudah dijelaskan. Non-linear/interaksi: Random Forest, Gradient Boosting, SVR kernel, NN. |

Faktor tambahan di praktik: **kebutuhan interpretasi (domain sensitif)**, **latensi (real time)**, dan **biaya komputasi**.

### 1.2 Cara membaca metrik
| Metrik | Rumus singkat | Arah baik | Catatan |
|---|---|---|---|
| MAE | rata-rata abs(y - ŷ) | makin kecil | Satuan sama dengan target; tahan terhadap outlier |
| MSE | rata-rata (y - ŷ)² | makin kecil | Satuan kuadrat; menghukum error besar |
| RMSE | akar MSE | makin kecil | Satuan sama dengan target |
| R² | 1 - SSE/SST | makin dekat 1 | 0 = sama dengan prediksi rata-rata; negatif = lebih buruk dari rata-rata |

**Jawaban untuk "MAE/MSE harus dekat atau kurang dari 1?"**
Tidak ada ambang universal. MAE dan MSE bergantung pada **skala target**. Error 5.000 rupiah itu besar jika target berkisar 10 ribu, tetapi kecil jika target berkisar 10 juta. Karena itu:
1. Bandingkan dengan **baseline** (selalu menebak rata-rata). Model berguna jika MAE/MSE-nya lebih kecil dari baseline.
2. Gunakan versi ternormalisasi: **RMSE / std(y)**. Nilai < 1 berarti lebih baik dari baseline, makin mendekati 0 makin baik.
3. Jika target sudah di-standardisasi (seperti pada notebook awal Anda), maka varians target = 1 sehingga baseline MSE ≈ 1. Di kondisi itu MSE < 1 berarti model lebih baik dari menebak rata-rata. Ini kebetulan akibat skala, bukan aturan umum.
4. R² dibaca sebaliknya: **makin dekat 1 makin baik**. Ambang "bagus" bergantung domain (data perilaku manusia sering R² 0,3-0,6 sudah wajar, data fisik/mesin biasanya > 0,9).

In [ ]:
import time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

# ===== KONFIGURASI =====
CSV_PATH   = "ecommerce_sales_customer_analytics_150k.csv"
TARGET     = "net_sales"
MAX_ROWS   = 30000   # batasi baris untuk perbandingan agar cepat; set None untuk semua data
ID_LIKE    = ["order_id", "customer_id", "customer_review"]   # kolom ID/teks bebas, dibuang
LEAK_THRESHOLD = 0.95  # fitur dengan |korelasi| > ini dengan target dicurigai kebocoran
RANDOM_STATE = 1

## 2. Memuat Data

In [ ]:
if Path(CSV_PATH).exists():
    df = pd.read_csv(CSV_PATH)
    print("Memakai data asli:", df.shape)
else:
    print("CSV tidak ditemukan -> memakai data SINTETIS untuk demo.")
    rng = np.random.default_rng(RANDOM_STATE); n = 20000
    age = rng.integers(18, 70, n); qty = rng.integers(1, 8, n)
    price = rng.lognormal(3.5, 0.6, n); disc = rng.choice([0, .05, .1, .2], n)
    chan = rng.choice(["web", "app", "marketplace", "offline"], n)
    cat = rng.choice(["fashion", "electronics", "home", "beauty"], n)
    status = rng.choice(["delivered", "returned", "cancelled"], n, p=[.85, .1, .05])
    gross = qty * price
    net = gross * (1 - disc) * np.where(status == "delivered", 1, 0.3) \
          + 0.02 * age * np.sqrt(price) + rng.normal(0, 5, n)
    df = pd.DataFrame(dict(order_id=np.arange(n), customer_age=age, quantity=qty, unit_price=price,
                           discount=disc, sales_channel=chan, category=cat, order_status=status,
                           gross_sales=gross, net_sales=net))
    df.loc[rng.choice(n, 300, replace=False), "customer_age"] = np.nan
df.head()

## 3. Pembersihan Dasar (tanpa kebocoran data)

In [ ]:
df = df.drop(columns=[c for c in ID_LIKE if c in df.columns])
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])                      # baris tanpa target tidak bisa dipakai
miss = df.isna().mean().sort_values(ascending=False)
print("Missing value (>0):\n", miss[miss > 0].round(4))
# kolom dengan > 40% kosong dibuang; sisanya diimputasi DI DALAM pipeline
df = df.drop(columns=miss[miss > 0.40].index.difference([TARGET]))
if MAX_ROWS and len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=RANDOM_STATE)
print("Bentuk data akhir:", df.shape)

## 4. Diagnosis 3 Faktor Penentu Algoritma
### 4.1 Jenis data

In [ ]:
num_cols = df.drop(columns=[TARGET]).select_dtypes(include="number").columns.tolist()
cat_cols = df.drop(columns=[TARGET]).select_dtypes(exclude="number").columns.tolist()
print("Fitur numerik   :", len(num_cols), num_cols)
print("Fitur kategorikal:", len(cat_cols), cat_cols)
print("\nKardinalitas kategorikal:")
print(df[cat_cols].nunique().sort_values(ascending=False) if cat_cols else "-")
print("\nSkewness target:", round(df[TARGET].skew(), 2), "(>1 = condong kanan, pertimbangkan log-transform / model tahan outlier)")

### 4.2 Ukuran dataset

In [ ]:
n, p = df.shape[0], df.shape[1] - 1
ratio = n / p
size_label = "KECIL (<1.000)" if n < 1000 else "MENENGAH (1.000-100.000)" if n < 100000 else "BESAR (>100.000)"
print(f"Baris={n:,} | Fitur={p} | Baris per fitur={ratio:,.0f} | Kategori ukuran: {size_label}")

### 4.3 Linearitas dan deteksi kebocoran

In [ ]:
numeric_all = df.select_dtypes(include="number")
pear = numeric_all.corr(method="pearson")[TARGET].drop(TARGET)
spear = numeric_all.corr(method="spearman")[TARGET].drop(TARGET)
lin = pd.DataFrame({"pearson": pear, "spearman": spear})
lin["selisih"] = (lin.spearman.abs() - lin.pearson.abs())
display(lin.sort_values("pearson", key=abs, ascending=False).round(3))
print("Selisih besar (spearman >> pearson) = hubungan monoton tapi tidak linear.")

leak = lin[lin.pearson.abs() > LEAK_THRESHOLD].index.tolist()
print("\nFitur dicurigai kebocoran (hampir sama dengan target):", leak or "tidak ada")

In [ ]:
# Koreksi kebocoran: fitur yang diturunkan langsung dari target (mis. gross_sales -> net_sales) dibuang
# agar model belajar pola sebenarnya. Hapus baris ini jika kolom tersebut memang tersedia saat prediksi.
df = df.drop(columns=leak)
num_cols = [c for c in num_cols if c not in leak]

## 5. Pembagian Data dan Preprocessing (Pipeline)

In [ ]:
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

num_pipe_scaled = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
num_pipe_raw    = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_onehot = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))])
cat_ordinal = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oe", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])

prep_linear = ColumnTransformer([("n", num_pipe_scaled, num_cols), ("c", cat_onehot, cat_cols)])   # butuh scaling
prep_tree   = ColumnTransformer([("n", num_pipe_raw, num_cols),    ("c", cat_ordinal, cat_cols)])  # tidak butuh scaling
# Scaler dan imputer di-fit HANYA pada data latih (di dalam pipeline) -> tidak ada kebocoran.

## 6. Perbandingan Banyak Algoritma Regresi

In [ ]:
models = {
    "Baseline (rata-rata)":     Pipeline([("p", prep_linear), ("m", DummyRegressor(strategy="mean"))]),
    "Linear Regression":        Pipeline([("p", prep_linear), ("m", LinearRegression())]),
    "Ridge (L2)":               Pipeline([("p", prep_linear), ("m", Ridge(alpha=1.0))]),
    "Lasso (L1)":               Pipeline([("p", prep_linear), ("m", Lasso(alpha=0.01))]),
    "ElasticNet (L1+L2)":       Pipeline([("p", prep_linear), ("m", ElasticNet(alpha=0.01, l1_ratio=0.5))]),
    "KNN Regressor":            Pipeline([("p", prep_linear), ("m", KNeighborsRegressor(n_neighbors=10, n_jobs=-1))]),
    "Decision Tree":            Pipeline([("p", prep_tree),   ("m", DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE))]),
    "Random Forest":            Pipeline([("p", prep_tree),   ("m", RandomForestRegressor(n_estimators=150, max_depth=14, n_jobs=-1, random_state=RANDOM_STATE))]),
    "Gradient Boosting":        Pipeline([("p", prep_tree),   ("m", GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=RANDOM_STATE))]),
    "HistGradientBoosting":     Pipeline([("p", prep_tree),   ("m", HistGradientBoostingRegressor(max_iter=300, random_state=RANDOM_STATE))]),
    "Neural Net (MLP)":         Pipeline([("p", prep_linear), ("m", MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True, random_state=RANDOM_STATE))]),
}

kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, mdl in models.items():
    t0 = time.time(); mdl.fit(X_train, y_train); fit_s = time.time() - t0
    t0 = time.time(); pred = mdl.predict(X_test); pred_ms = (time.time() - t0) / len(X_test) * 1000
    mse = mean_squared_error(y_test, pred)
    cv = cross_val_score(mdl, X_train, y_train, cv=kf, scoring="r2", n_jobs=1)
    rows.append(dict(Model=name, MAE=mean_absolute_error(y_test, pred), MSE=mse, RMSE=np.sqrt(mse),
                     nRMSE=np.sqrt(mse) / y_test.std(), R2=r2_score(y_test, pred),
                     CV_R2_mean=cv.mean(), CV_R2_std=cv.std(), Fit_detik=fit_s, Pred_ms_per_baris=pred_ms))
    print(f"selesai: {name}")
res = pd.DataFrame(rows).set_index("Model").sort_values("RMSE")
res.round(4)

**Cara membaca tabel**
- `nRMSE` = RMSE / std(y). **< 1** berarti lebih baik dari baseline, **mendekati 0** makin baik.
- `R2` makin dekat 1 makin baik; **negatif** = lebih buruk dari baseline.
- `CV_R2_std` kecil = model stabil antar-fold. Jika `R2` test jauh di bawah `CV_R2_mean` curigai overfit atau data shift.
- `Pred_ms_per_baris` dipakai untuk menilai kelayakan **real time**.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
res["R2"].sort_values().plot.barh(ax=ax[0], color="#4C78A8"); ax[0].set_title("R2 (makin dekat 1 makin baik)")
res["RMSE"].sort_values(ascending=False).plot.barh(ax=ax[1], color="#E4572E"); ax[1].set_title("RMSE (makin kecil makin baik)")
res["Pred_ms_per_baris"].sort_values(ascending=False).plot.barh(ax=ax[2], color="#54A24B"); ax[2].set_title("Latensi prediksi (ms/baris)")
plt.tight_layout(); plt.show()

best = res.index[0] if res.index[0] != "Baseline (rata-rata)" else res.index[1]
pred = models[best].predict(X_test)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_test, pred, s=4, alpha=.3); lim = [y_test.min(), y_test.max()]
ax[0].plot(lim, lim, "r--"); ax[0].set(title=f"Aktual vs Prediksi ({best})", xlabel="Aktual", ylabel="Prediksi")
sns.histplot(y_test - pred, kde=True, ax=ax[1]); ax[1].set_title("Distribusi residual (idealnya simetris di sekitar 0)")
plt.tight_layout(); plt.show()

## 7. Ringkasan Otomatis: Apa yang Disarankan Data Anda?

In [ ]:
lin_r2 = res.loc["Ridge (L2)", "R2"]
nonlin_r2 = res.loc[["Random Forest", "Gradient Boosting", "HistGradientBoosting"], "R2"].max()
gap = nonlin_r2 - lin_r2
print(f"Ukuran data     : {size_label}")
print(f"Komposisi fitur : {len(num_cols)} numerik, {len(cat_cols)} kategorikal")
print(f"R2 linear (Ridge) = {lin_r2:.3f} | R2 terbaik non-linear = {nonlin_r2:.3f} | selisih = {gap:.3f}")
if gap > 0.05:
    print("-> Hubungan cenderung NON-LINEAR / ada interaksi. Pilih model berbasis pohon (RF / Boosting).")
else:
    print("-> Hubungan cenderung LINEAR. Model linear teregularisasi sudah memadai dan lebih mudah dijelaskan.")
print(f"Model terbaik berdasarkan RMSE: {best}")

## 8. Daftar Algoritma Regresi dan Peruntukannya

In [ ]:
panduan = pd.DataFrame([
 ["Linear Regression (OLS)", "Hubungan linear, fitur tidak saling berkorelasi kuat", "Kecil-besar (sangat cepat)", "Ya, sangat cepat", "Sangat baik (koefisien jelas)", "Baseline wajib; keuangan/medis/kebijakan yang perlu penjelasan"],
 ["Ridge (L2)", "Linear dengan multikolinearitas, fitur banyak", "Kecil-besar", "Ya", "Baik", "Menstabilkan koefisien; data dengan fitur saling berkorelasi"],
 ["Lasso (L1)", "Linear dengan banyak fitur tidak relevan", "Kecil-besar", "Ya", "Baik (seleksi fitur otomatis)", "Seleksi fitur; model ringkas"],
 ["ElasticNet", "Gabungan Ridge+Lasso, fitur berkelompok", "Kecil-besar", "Ya", "Baik", "Data dimensi tinggi dengan fitur berkorelasi"],
 ["Polynomial Regression", "Non-linear sederhana dan terbatas", "Kecil-menengah", "Ya (derajat rendah)", "Sedang", "Kurva halus 1-3 fitur; hati-hati overfit"],
 ["KNN Regressor", "Non-linear lokal, dimensi rendah", "Kecil-menengah (lambat saat data besar)", "Kurang (prediksi menyimpan seluruh data)", "Sedang", "Rekomendasi berbasis kemiripan; prototipe cepat"],
 ["Decision Tree", "Non-linear, data campuran", "Kecil-besar", "Ya", "Baik (aturan if-else)", "Aturan yang bisa dibaca bisnis; mudah overfit bila dalam"],
 ["Random Forest", "Non-linear + interaksi, outlier, data campuran", "Menengah-besar", "Cukup (model besar, lebih lambat dari linear)", "Sedang (feature importance)", "Pilihan kuat default untuk data tabular tanpa banyak tuning"],
 ["Gradient Boosting / XGBoost / LightGBM / CatBoost", "Non-linear kompleks, data tabular", "Menengah-sangat besar", "Ya (inferensi cepat jika dioptimasi)", "Sedang (SHAP membantu)", "Sering akurasi terbaik di data tabular: harga, permintaan, risiko, penjualan"],
 ["SVR (kernel)", "Non-linear, fitur dimensi tinggi, data kecil", "Kecil saja (skala kuadratik)", "Kurang pada data besar", "Rendah", "Dataset kecil dengan pola halus (sensor, kimia)"],
 ["Neural Network (MLP)", "Non-linear sangat kompleks, data sangat besar", "Besar-sangat besar", "Ya setelah dilatih", "Rendah (black box)", "Data besar dan multimodal (gabungan teks/gambar/tabular)"],
 ["Gaussian Process", "Perlu estimasi ketidakpastian", "Sangat kecil", "Tidak", "Sedang", "Eksperimen mahal, optimasi Bayesian"],
 ["Quantile / Huber Regression", "Outlier kuat atau perlu rentang prediksi", "Kecil-besar", "Ya", "Baik", "Prediksi interval, data dengan error ekor berat"],
 ["Time-series (ARIMA, Prophet, dll.)", "Data berurutan waktu dan musiman", "Kecil-menengah", "Ya", "Baik", "Peramalan penjualan harian/mingguan"],
], columns=["Algoritma", "Cocok untuk (linearitas/jenis data)", "Skala data", "Real-time", "Interpretabilitas (domain sensitif)", "Contoh peruntukan"])
pd.set_option("display.max_colwidth", None)
panduan

## 9. Catatan Kritis untuk Notebook Anda Sebelumnya
1. **Standardisasi sebelum split** membuat statistik data uji ikut terhitung (kebocoran ringan). Di notebook ini scaler ada di dalam *pipeline*.
2. **Target ikut distandardisasi** sehingga MAE/MSE tidak lagi dalam satuan asli dan sulit diinterpretasi bisnis. Di sini target dibiarkan asli dan dilaporkan juga `nRMSE`.
3. **Penghapusan outlier dengan IQR pada semua kolom numerik** (termasuk target) dapat membuang transaksi besar yang justru penting untuk prediksi penjualan. Pertimbangkan capping, log-transform, atau model tahan outlier.
4. **Label Encoding pada kategori nominal** memberi urutan semu pada model linear/KNN. Dipakai One-Hot untuk model tersebut; Ordinal hanya untuk model pohon.
5. **Fitur turunan target** (mis. `gross_sales`, diskon) bisa membuat R² sangat tinggi secara menipu. Periksa bagian 4.3.
6. Penjelasan di bagian bawah notebook lama menyebut `LinearRegression`, capping IQR, dan `get_dummies`, padahal kode memakai RandomForest, penghapusan outlier, dan LabelEncoder. Sesuaikan teks dengan kode.

## 10. Referensi (publikasi 2021-2022, dalam 5 tahun terakhir)
Dikonfirmasi lewat pencarian saat penyusunan:
- Hodson, T. O. (2022). *Root-mean-square error (RMSE) or mean absolute error (MAE): when to use them or not.* Geoscientific Model Development, 15, 5481-5487. https://doi.org/10.5194/gmd-15-5481-2022 (Tidak ada metrik yang inheren lebih baik; RMSE optimal untuk error Gaussian, MAE untuk error Laplace.)
- Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS 2022 (Datasets and Benchmarks). https://arxiv.org/abs/2207.08815 (Model berbasis pohon masih unggul pada data tabular berukuran menengah, sekitar 10 ribu sampel.)

Dari ingatan, mohon diverifikasi sebelum dikutip:
- Shwartz-Ziv, R., & Armon, A. (2022). *Tabular data: Deep learning is not all you need.* Information Fusion, 81, 84-90.
- Borisov, V., et al. (2022). *Deep neural networks and tabular data: A survey.* IEEE Transactions on Neural Networks and Learning Systems.
- Chicco, D., Warrens, M. J., & Jurman, G. (2021). *The coefficient of determination R-squared is more informative than SMAPE, MAE, MAPE, MSE and RMSE in regression analysis evaluation.* PeerJ Computer Science, 7, e623.